####  Reference Documentation

In [126]:
# https://github.com/MicrosoftDocs/azure-ai-docs/blob/main/articles/ai-services/language-service/custom-named-entity-recognition/how-to/call-api.md#tab/client
# https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/textanalytics/azure-ai-textanalytics/samples/sample_recognize_custom_entities.py
# https://learn.microsoft.com/en-us/python/api/overview/azure/ai-textanalytics-readme?view=azure-python
# https://learn.microsoft.com/en-us/azure/ai-services/language-service/summarization/quickstart?tabs=text-summarization%2Cwindows&pivots=programming-language-python

####  Loading Libraries and clients

In [84]:
import os
import json
import time
import openai
import requests
import tiktoken
import pandas as pd
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from langchain.text_splitter import MarkdownHeaderTextSplitter
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, ContentFormat
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, DocumentAnalysisFeature

from azure.ai.textanalytics import (
    TextAnalyticsClient,
    ExtractiveSummaryAction,
    AbstractiveSummaryAction,
) 

from dotenv import load_dotenv
from openai import AzureOpenAI
load_dotenv(override=True)

aoai_client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
  api_version="2024-07-01-preview"
)

AZURE_LANGUAGE_ENDPOINT = os.environ["AZURE_LANGUAGE_ENDPOINT"]
AZURE_LANGUAGE_KEY = os.environ["AZURE_LANGUAGE_KEY"]
CUSTOM_ENTITIES_PROJECT_NAME = os.environ["CUSTOM_ENTITIES_PROJECT_NAME"]
CUSTOM_ENTITIES_DEPLOYMENT_NAME = os.environ["CUSTOM_ENTITIES_DEPLOYMENT_NAME"]

# Text Analytics Client
text_analytics_client = TextAnalyticsClient(
    endpoint=AZURE_LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_LANGUAGE_KEY),
)

# Document Intelligence Client
AZURE_DOC_INTELLIGENCE_ENDPOINT = os.environ["AZURE_DOC_INTELLIGENCE_ENDPOINT"]
AZURE_DOC_INTELLIGENCE_KEY = os.environ["AZURE_DOC_INTELLIGENCE_KEY"]
document_intelligence_client = DocumentIntelligenceClient(endpoint=AZURE_DOC_INTELLIGENCE_ENDPOINT, credential=AzureKeyCredential(AZURE_DOC_INTELLIGENCE_KEY), api_version="2024-07-31-preview")

# Azure OpenAI Client
aoai_client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  api_key=os.getenv("AZURE_OPENAI_API_KEY"),  
  api_version="2024-07-01-preview"
)

#### Document Intelligence OCR

In [85]:
def text_html_processing(OcrExtractionDIOutput):
    offset = 0
    page_map = []
    page_map_dict =[]

    for page_num, page in enumerate(OcrExtractionDIOutput.pages):
        tables_on_page = [
            table
            for table in (OcrExtractionDIOutput.tables or [])
            if table.bounding_regions and table.bounding_regions[0].page_number == page_num + 1
        ]
        #print(tables_on_page)

        # mark all positions of the table spans in the page
        page_offset = page.spans[0].offset
        page_length = page.spans[0].length
        table_chars = [-1] * page_length
        for table_id, table in enumerate(tables_on_page):
            for span in table.spans:
                # replace all table spans with "table_id" in table_chars array
                for i in range(span.length):
                    idx = span.offset - page_offset + i
                    if idx >= 0 and idx < page_length:
                        table_chars[idx] = table_id

        # build page text by replacing characters in table spans with table html
        page_text = ""
        added_tables = set()
        for idx, table_id in enumerate(table_chars):
            if table_id == -1:
                page_text += OcrExtractionDIOutput.content[page_offset + idx]
            elif table_id not in added_tables:
                page_text += table_to_html(tables_on_page[table_id])
                added_tables.add(table_id)

        page_text += " "
        page_map.append((page_num+1, offset, page_text))

        single_page_dict = {}
        single_page_dict['page_num']= page_num+1
        single_page_dict['content'] = page_text
        single_page_dict['offset'] = offset
        page_map_dict.append(single_page_dict)

        offset += len(page_text)

    return page_map_dict

In [86]:
def OcrExtractionDI(relative_path: str, Markdown: [bool]=True):
    
    path_to_document = os.path.abspath(
        os.path.join(relative_path))
    
    if Markdown==True:
        output_format = ContentFormat.MARKDOWN
    else:
        output_format = None

    with open(path_to_document, "rb") as f:
        poller = document_intelligence_client.begin_analyze_document("prebuilt-layout", 
                                                                    analyze_request=f, content_type="application/octet-stream", 
                                                                    output_content_format=output_format)
    OcrExtractionDIOutput = poller.result()
    
    if Markdown==False:
        pagemap = text_html_processing(OcrExtractionDIOutput)
        extracted_processed_text = pagemap
    else:
        extracted_processed_text = OcrExtractionDIOutput

    return extracted_processed_text

In [87]:
def MdFormatting(ocr_extraction):
    doc_string = ocr_extraction.content

    ## Split the document into chunks base on markdown headers.
    headers_to_split_on = [
        ("#", "Title"),
        ("##", "Header 1"),
        ("###", "Header 2"),
        ("####", "Header 3"),
        ("#####", "Header 4"),
        ("######", "Header 5"),
        ("#######", "Header 6"),
    ]
    text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    markdown_chunks = text_splitter.split_text(doc_string)

    chunk_list = []
    for chunk in markdown_chunks:
        try:
            title = chunk.metadata['Title']
        except:
            title = ""
        try:
            header1 = chunk.metadata['Header 1']
        except:
            header1 = ""
        try:
            header2 = chunk.metadata['Header 2']
        except:
            header2 = ""
        try:
            header3 = chunk.metadata['Header 3']
        except:
            header3 = ""    

        chunk_list.append({"title": title,"header_1":header1,"header_2":header2,"header_3":header3,"content": chunk.page_content})
    return pd.DataFrame(chunk_list)

In [88]:
relative_path = "data/raw/"
Filename = "10Q-MSFT-04-25-2023.pdf"

Markdown = True

ocr_extraction = OcrExtractionDI(relative_path = relative_path+Filename, Markdown=Markdown)
content_map = MdFormatting(ocr_extraction)
content_map.groupby(by=["title",'header_1',"header_2"],sort=False)['content'].count().reset_index()[:30]

,title,header_1,header_2,content
0,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,FORM 10-Q,,1
1,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,MICROSOFT CORPORATION,,1
2,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,MICROSOFT CORPORATION FORM 10-Q For the Quarte...,,1
3,PART I. FINANCIAL INFORMATION ITEM 1. FINANCIA...,,,1
4,PART I. FINANCIAL INFORMATION ITEM 1. FINANCIA...,COMPREHENSIVE INCOME STATEMENTS,,1
5,PART ! Item 1,BALANCE SHEETS,,1
6,PART ! Item 1,BALANCE SHEETS,CASH FLOWS STATEMENTS,1
7,STOCKHOLDERS' EQUITY STATEMENTS,,,1
8,NOTES TO FINANCIAL STATEMENTS (Unaudited),NOTE 1 - ACCOUNTING POLICIES,Accounting Principles,1
9,NOTES TO FINANCIAL STATEMENTS (Unaudited),NOTE 1 - ACCOUNTING POLICIES,Principles of Consolidation,1


In [97]:
print(ocr_extraction.content)

# UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549


## FORM 10-Q

☒
QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the Quarterly Period Ended December 31, 2022
OR

☐
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the Transition Period From
to

Commission File Number 001-37845


## MICROSOFT CORPORATION

WASHINGTON
(STATE OF INCORPORATION)
ONE MICROSOFT WAY, REDMOND, WASHINGTON 98052-6399
(425) 882-8080
www.microsoft.com/investor

91-1144442
(I.R.S. ID)


<table>
<tr>
<th colspan="3">Securities registered pursuant to Section 12(b) of the Act:</th>
</tr>
<tr>
<th>Title of each class</th>
<th>Trading Symbol</th>
<th>Name of exchange on which registered</th>
</tr>
<tr>
<td>Common stock, $0.00000625 par value per share</td>
<td>MSFT</td>
<td>NASDAQ</td>
</tr>
<tr>
<td>3.125% Notes due 2028</td>
<td>MSFT</td>
<td>NASDAQ</td>
</tr>
<tr>
<td>2.625% Notes due 2033</td>
<td>MSFT</td

In [89]:
relative_path = "data/raw/"
Filename = "10Q-MSFT-01-24-2023.pdf"

Markdown = True

ocr_extraction = OcrExtractionDI(relative_path = relative_path+Filename, Markdown=Markdown)
content_map = MdFormatting(ocr_extraction)
content_map.groupby(by=["title",'header_1',"header_2"],sort=False)['content'].count().reset_index()[:30]

,title,header_1,header_2,content
0,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,FORM 10-Q,,1
1,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,MICROSOFT CORPORATION,,1
2,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,MICROSOFT CORPORATION FORM 10-Q For the Quarte...,,1
3,PART I. FINANCIAL INFORMATION ITEM 1. FINANCIA...,,,1
4,PART I. FINANCIAL INFORMATION ITEM 1. FINANCIA...,COMPREHENSIVE INCOME STATEMENTS,,1
5,PART ! Item 1,BALANCE SHEETS,,1
6,CASH FLOWS STATEMENTS,,,1
7,STOCKHOLDERS' EQUITY STATEMENTS,,,1
8,NOTES TO FINANCIAL STATEMENTS (Unaudited),NOTE 1 - ACCOUNTING POLICIES,Accounting Principles,1
9,NOTES TO FINANCIAL STATEMENTS (Unaudited),NOTE 1 - ACCOUNTING POLICIES,Principles of Consolidation,1


In [102]:
content_map.loc[3]

title        PART I. FINANCIAL INFORMATION ITEM 1. FINANCIA...
header_1                                                      
header_2                                                      
header_3                                                      
content      <table>\n<tr>\n<th>(In millions, except per sh...
token_len                                                 1091
Name: 3, dtype: object

In [101]:
print(content_map.content[3])

<table>
<tr>
<th>(In millions, except per share amounts) (Unaudited)</th>
<th colspan="3">Three Months Ended December 31,</th>
<th>Six Months Ended December 31,</th>
</tr>
<tr>
<th></th>
<th>2022</th>
<th>2021</th>
<th>2022</th>
<th>2021</th>
</tr>
<tr>
<td>Revenue:</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td>Product</td>
<td>$ 16,517</td>
<td>$ 20,779</td>
<td>$ 32,258</td>
<td>$ 37,410</td>
</tr>
<tr>
<td>Service and other</td>
<td>36,230</td>
<td>30,949</td>
<td>70,611</td>
<td>59,635</td>
</tr>
<tr>
<td>Total revenue</td>
<td>52,747</td>
<td>51,728</td>
<td>102,869</td>
<td>97,045</td>
</tr>
<tr>
<td>Cost of revenue:</td>
<td></td>
<td></td>
<td></td>
<td></td>
</tr>
<tr>
<td>Product</td>
<td>5,690</td>
<td>6,331</td>
<td>9,992</td>
<td>10,123</td>
</tr>
<tr>
<td>Service and other</td>
<td>11,798</td>
<td>10,629</td>
<td>22,948</td>
<td>20,483</td>
</tr>
<tr>
<td>Total cost of revenue</td>
<td>17,488</td>
<td>16,960</td>
<td>32,940</td>
<td>30,606</td>
</tr>
<tr>
<t

In [92]:
def num_tokens_from_string(string: str) -> int:
    encoding = tiktoken.encoding_for_model("gpt-4o")
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [94]:
content_map['token_len'] = content_map.content.apply(num_tokens_from_string)

In [95]:
content_map

,title,header_1,header_2,header_3,content,token_len
0,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,FORM 10-Q,,,☒\nQUARTERLY REPORT PURSUANT TO SECTION 13 OR ...,96
1,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,MICROSOFT CORPORATION,,,WASHINGTON\n(STATE OF INCORPORATION)\nONE MICR...,692
2,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,MICROSOFT CORPORATION FORM 10-Q For the Quarte...,,,"<table>\n<tr>\n<th>PART I.</th>\n<th colspan=""...",670
3,PART I. FINANCIAL INFORMATION ITEM 1. FINANCIA...,,,,"<table>\n<tr>\n<th>(In millions, except per sh...",1091
4,PART I. FINANCIAL INFORMATION ITEM 1. FINANCIA...,COMPREHENSIVE INCOME STATEMENTS,,,<table>\n<tr>\n<th>(In millions) (Unaudited)</...,396
...,...,...,...,...,...,...
130,ITEM 6. EXHIBITS,15.1 Letter regarding unaudited interim financ...,,,31.1\nCertification of Chief Executive Officer...,282
131,SIGNATURE,,,,Pursuant to the requirements of the Securities...,323
132,CERTIFICATION,,,,"I, Satya Nadella, certify that: \n1\. I have ...",655
133,CERTIFICATION,CERTIFICATION,,,"I, Amy E. Hood, certify that: \n1\. I have re...",660


#### Using Summarization API